# AIC 2026 B01 cloud feature extraction

This notebook is a thin launcher for the tested ZIP-streamed CLI. It never extracts the keyframe or video ZIPs.

In [ ]:
import csv
import hashlib
import io
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

BATCH_ID = 'aic25-b1-reference'
EXPECTED_VIDEOS = 873
EXPECTED_MAPPING_ROWS = 177_321
EXPECTED_CANONICAL_FRAMES = 176_707
DRIVE_ROOT = Path('/content/drive/MyDrive/AIC2026/B01')
REPO_ROOT = Path('/content/aic2026')
RAW_ROOT = DRIVE_ROOT / 'raw'
OUTPUT_ROOT = DRIVE_ROOT / 'out'
SCRATCH_ROOT = Path('/content/aic2026-scratch')
SOURCE_BUNDLE = DRIVE_ROOT / 'source' / 'aic2026-colab-source.zip'

assert SOURCE_BUNDLE.is_file(), f'Missing source bundle: {SOURCE_BUNDLE}'
REPO_ROOT.mkdir(parents=True, exist_ok=True)
repo_root = REPO_ROOT.resolve()
with zipfile.ZipFile(SOURCE_BUNDLE) as source_archive:
    for member in source_archive.infolist():
        destination = (REPO_ROOT / member.filename).resolve()
        if destination != repo_root and repo_root not in destination.parents:
            raise RuntimeError(f'Unsafe source bundle member: {member.filename}')
        if member.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        with source_archive.open(member) as source, destination.open('wb') as target:
            shutil.copyfileobj(source, target, length=1024 * 1024)

assert (REPO_ROOT / 'pyproject.toml').is_file(), 'Source bundle is incomplete'
assert (RAW_ROOT / 'map-keyframes-aic25-b1.zip').is_file()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)

gpu_name = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name', '--format=csv,noheader'
], text=True).splitlines()[0].strip()
if not any(model in gpu_name.upper() for model in ('T4', 'A100')):
    raise RuntimeError(f'T4/A100 required; allocated GPU is {gpu_name!r}')
ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
disk_gib = shutil.disk_usage('/content').free / 1024**3
if disk_gib < 20:
    raise RuntimeError(f'At least 20 GiB ephemeral disk is required; found {disk_gib:.1f}')
print(json.dumps({
    'gpu': gpu_name, 'ram_gib': round(ram_gib, 1),
    'high_ram': ram_gib >= 20, 'ephemeral_disk_free_gib': round(disk_gib, 1),
}, ensure_ascii=False))

## Locked environment and pinned models

The commands install Python 3.11, sync the locked competition group, install ffmpeg, and acquire only the three pinned model families used here.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.11.16'], check=True)
subprocess.run(['uv', 'python', 'install', '3.11'], check=True)
subprocess.run(['uv', 'sync', '--python', '3.11', '--group', 'competition', '--frozen'], check=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
for model in ('visual', 'reference-ocr', 'reference-asr'):
    subprocess.run(['uv', 'run', '--frozen', 'python', 'scripts/prepare_models.py', '--only', model], check=True)


In [ ]:
keyframe_zips = sorted(RAW_ROOT.glob('Keyframes_L*.zip'))
video_zips = sorted(RAW_ROOT.glob('Videos_L*.zip'))
expected_keyframes = {
    'Keyframes_L21.zip', 'Keyframes_L22.zip', 'Keyframes_L23.zip',
    'Keyframes_L24.zip', 'Keyframes_L25.zip',
    *(f'Keyframes_L26_{part}.zip' for part in 'abcde'),
    'Keyframes_L27.zip', 'Keyframes_L28.zip', 'Keyframes_L29.zip',
    'Keyframes_L30.zip',
}
expected_videos = {
    *(f'Videos_L{group}_a.zip' for group in (21, 22, 23, 24, 25, 27, 28, 29, 30)),
    *(f'Videos_L26_{part}.zip' for part in 'abcde'),
}
assert {path.name for path in keyframe_zips} == expected_keyframes
assert {path.name for path in video_zips} == expected_videos

keyframe_video_ids = set()
keyframe_rows = 0
for archive_path in keyframe_zips:
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            if Path(member.filename).suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
                keyframe_rows += 1
                keyframe_video_ids.add(Path(member.filename).parent.name)

video_ids = set()
for archive_path in video_zips:
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            if Path(member.filename).suffix.lower() in {'.mp4', '.mkv', '.avi'}:
                video_ids.add(Path(member.filename).stem)

mapping_rows = 0
mapping_video_ids = set()
canonical_identities = set()
with zipfile.ZipFile(RAW_ROOT / 'map-keyframes-aic25-b1.zip') as map_archive:
    for member in map_archive.infolist():
        if Path(member.filename).suffix.lower() != '.csv':
            continue
        video_id = Path(member.filename).stem
        mapping_video_ids.add(video_id)
        with map_archive.open(member) as stream:
            reader = csv.DictReader(io.TextIOWrapper(stream, encoding='utf-8-sig', newline=''))
            for row in reader:
                mapping_rows += 1
                canonical_identities.add((video_id, int(row['frame_idx'])))

assert keyframe_rows == EXPECTED_MAPPING_ROWS
assert mapping_rows == EXPECTED_MAPPING_ROWS
assert len(canonical_identities) == EXPECTED_CANONICAL_FRAMES
assert len(video_ids) == EXPECTED_VIDEOS
assert keyframe_video_ids == video_ids == mapping_video_ids
print(json.dumps({
    'keyframe_archives': len(keyframe_zips), 'video_archives': len(video_zips),
    'videos': len(video_ids), 'mapping_rows': mapping_rows,
    'canonical_frames': len(canonical_identities),
}, ensure_ascii=False))

keyframe_args = [part for path in keyframe_zips for part in ('--keyframes-zip', str(path))]
video_args = [part for path in video_zips for part in ('--videos-zip', str(path))]
common = [
    'uv', 'run', '--frozen', 'python', 'scripts/colab_provision_zip.py',
    '--maps-zip', str(RAW_ROOT / 'map-keyframes-aic25-b1.zip'),
    *keyframe_args, '--output-dir', str(OUTPUT_ROOT),
    '--batch-id', BATCH_ID,
    '--expected-videos', str(EXPECTED_VIDEOS),
    '--expected-frames', str(EXPECTED_CANONICAL_FRAMES),
    '--progress-file', str(OUTPUT_ROOT / 'cloud_work' / 'progress.json'),
    '--resume',
]

## Publish SigLIP2 immediately

Existing 873 SigLIP2 receipts are finalized as a vector-only schema-v3 bundle. OCR runs separately on partition workers.

In [ ]:
subprocess.run(common + [
    '--finalize-only', '--finalize-channels', 'siglip2'
], check=True)
subprocess.run([
    'uv', 'run', '--frozen', 'python', 'scripts/import_cloud_artifacts.py',
    '--artifact-dir', str(OUTPUT_ROOT), '--batch-id', BATCH_ID,
    '--channels', 'siglip2', '--validate-only'
], check=True)
subprocess.run([
    'uv', 'run', '--frozen', 'python', 'scripts/package_cloud_artifacts.py',
    '--artifact-dir', str(OUTPUT_ROOT), '--output-dir', str(OUTPUT_ROOT / 'bundles'),
    '--batch-id', BATCH_ID, '--channels', 'siglip2'
], check=True)


## ASR after both OCR workers finish

Each zipped video is piped to ffmpeg; only one temporary mono WAV is present at a time. Faster-Whisper VAD is always enabled.

In [ ]:
subprocess.run(common + video_args + [
    '--stages', 'asr', '--device', 'auto', '--ffmpeg-path', 'ffmpeg',
    '--scratch-dir', str(SCRATCH_ROOT)
], check=True)


## Atomic finalization and offline verification

In [ ]:
subprocess.run(common + [
    '--finalize-only', '--finalize-channels', 'siglip2,ocr,asr'
], check=True)
subprocess.run([
    'uv', 'run', '--frozen', 'python', 'scripts/import_cloud_artifacts.py',
    '--artifact-dir', str(OUTPUT_ROOT), '--batch-id', BATCH_ID, '--validate-only'
], check=True)

artifact_names = (
    'b01_siglip2_features.npy', 'b01_siglip2_manifest.json',
    'b01_ocr_features.jsonl.gz', 'b01_asr_transcripts.jsonl.gz',
)
for name in artifact_names:
    path = OUTPUT_ROOT / name
    print(name, path.stat().st_size)

manifest_path = OUTPUT_ROOT / 'b01_siglip2_manifest.json'
manifest_sha256 = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
bundle_dir = OUTPUT_ROOT / 'bundles'
bundle_dir.mkdir(parents=True, exist_ok=True)
bundle_path = bundle_dir / f'b01_cloud_artifacts_{manifest_sha256[:12]}.zip'
bundle_tmp = bundle_path.with_suffix('.zip.tmp')
with zipfile.ZipFile(bundle_tmp, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as bundle:
    for name in artifact_names:
        bundle.write(OUTPUT_ROOT / name, arcname=name)
os.replace(bundle_tmp, bundle_path)
bundle_digest = hashlib.sha256()
with bundle_path.open('rb') as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        bundle_digest.update(chunk)
bundle_sha256 = bundle_digest.hexdigest()
sidecar_path = bundle_path.with_suffix(bundle_path.suffix + '.sha256')
sidecar_path.write_text(f'{bundle_sha256}  {bundle_path.name}\n', encoding='ascii')
print(json.dumps({
    'manifest_sha256': manifest_sha256, 'bundle': str(bundle_path),
    'bundle_sha256': bundle_sha256, 'sha256_sidecar': str(sidecar_path),
}, ensure_ascii=False))